# IMDB Sentiment Classification with PyTorch Transformer

In [14]:
import math
import random

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# Load and Inspect the dataset

In [3]:
# Load the IMBD data 

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
sample = dataset["train"][0]

print("Review:")
print(sample["text"])

print("\nLabel:")
print(sample["label"])

Review:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betw

In [5]:
print(len(dataset['train']))
print(len(dataset['test']))

25000
25000


In [6]:
def tokenize(text):
    return text.lower().split()

In [7]:
text = "This movie was absolutely amazing!"

tokens = tokenize(text)

print(tokens)

['this', 'movie', 'was', 'absolutely', 'amazing!']


# Build vocabulary

In [8]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_ID = 0
UNK_ID = 1

MAX_VOCAB_SIZE = 20000

counter = Counter()

for example in dataset["train"]:
    counter.update(tokenize(example["text"]))

most_common_tokens = counter.most_common(MAX_VOCAB_SIZE - 2)

vocab = {
    PAD_TOKEN: PAD_ID,
    UNK_TOKEN: UNK_ID
}

for idx, (token, _) in enumerate(most_common_tokens, start=2):
    vocab[token] = idx

print("Vocabulary size:", len(vocab))

Vocabulary size: 20000


In [10]:
# token --> id   
def numericalize(tokens):
    return [
        vocab.get(token, UNK_ID)
        for token in tokens
    ]

In [11]:
tokens = tokenize("this movie was amazing")

token_ids = numericalize(tokens)

print("Tokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

Tokens:
['this', 'movie', 'was', 'amazing']

Token IDs:
[10, 20, 14, 531]


# Dataset preprocessing

In [12]:
MAX_SEQ_LEN = 256

def encode_text(text):
    tokens = tokenize(text)

    token_ids = numericalize(tokens)

    token_ids = token_ids[:MAX_SEQ_LEN]

    return token_ids

In [13]:
text = "This movie was absolutely amazing and enjoyable."

encoded = encode_text(text)

print(encoded)
print("Length:", len(encoded))

[10, 20, 14, 371, 531, 4, 3671]
Length: 7


# Data Loader

In [15]:
# Collate function
# Different reviews have different lengths, so each batch needs padding.

def collate_batch(batch):
    texts = [torch.tensor(encode_text(item['text']),dtype = torch.long) 
             for item in batch]

    labels = torch.tensor(
        [item['label'] for item in batch],dtype = torch.long
    )

    input_ids = pad_sequence(
        texts,
        batch_first= True,
        padding_value= PAD_ID
    )

    return input_ids,labels

In [16]:
train_data = dataset["train"]
test_data = dataset["test"]

BATCH_SIZE = 32

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

In [17]:
input_ids, labels = next(iter(train_loader))

print("Input shape:", input_ids.shape)
print("Labels shape:", labels.shape)

print("\nInput IDs:")
print(input_ids[:2])

print("\nLabels:")
print(labels[:2])

Input shape: torch.Size([32, 256])
Labels shape: torch.Size([32])

Input IDs:
tensor([[    9,   366,   148,    10,   528,    29,   394,    19,   328,     4,
           480,  2893,   229,   179,     3,  7999,   323,    14,    38,    76,
            15,     3,     1,     8,   795,  6663,     1,     4,    99,    49,
         11186,   449,   338,    57,  1133,    10,  1881,    19,     3,  2246,
          1112,    11,    14,    99,  1267,   285,   343,   205,    11,    39,
          3428,    16,     2,     1,     9,    62,     2,   673,     6, 13440,
            10,   151,     4,   236,    11,    12,   126,  1370,    65,   168,
           310,   281,    44,    23,   168,     2,  1306,    11,  2310,    62,
             1,    13,   165,    26,     6,  6565,     6,     3,   721, 11315,
           629,    45,  2141, 16876,   684,    15, 10916,     1,   498,    15,
             1,   145,    23,   550,  1411,     9,    14,     3,     1,     5,
             2,  3184,  2063,  1387,    31,  1675,   

In [18]:
# positional encoding 

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        position = torch.arange(
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        seq_len = x.size(1)

        return x + self.pe[:, :seq_len]

In [19]:
class TransformerClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=128,
        num_heads=8,
        num_layers=2,
        d_ff=256,
        num_classes=2,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_ID
        )

        self.positional_encoding = PositionalEncoding(
            d_model
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.classifier = nn.Linear(
            d_model,
            num_classes
        )

    def forward(self, input_ids):

        padding_mask = input_ids.eq(PAD_ID)

        x = self.embedding(input_ids)

        x = x * math.sqrt(
            self.embedding.embedding_dim
        )

        x = self.positional_encoding(x)

        x = self.transformer_encoder(
            x,
            src_key_padding_mask=padding_mask
        )

        valid_tokens = (~padding_mask).unsqueeze(-1)

        x = x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

        pooled = x.sum(dim=1) / valid_tokens.sum(
            dim=1
        ).clamp(min=1)

        logits = self.classifier(pooled)

        return logits

In [20]:
model = TransformerClassifier(
    vocab_size=len(vocab),
    d_model=128,
    num_heads=8,
    num_layers=2,
    d_ff=256,
    num_classes=2
)

model = model.to(device)

print(model)

TransformerClassifier(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (positional_encoding): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [21]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 2825218
Trainable parameters: 2825218


In [22]:
input_ids, labels = next(iter(train_loader))

input_ids = input_ids.to(device)
labels = labels.to(device)

logits = model(input_ids)

print("Input shape:", input_ids.shape)
print("Logits shape:", logits.shape)

Input shape: torch.Size([32, 256])
Logits shape: torch.Size([32, 2])


In [23]:
print(logits[:5])

tensor([[-0.1276,  0.2205],
        [-0.1812,  0.1501],
        [-0.1744,  0.1631],
        [-0.1245,  0.1146],
        [-0.0363,  0.2132]], device='cuda:0', grad_fn=<SliceBackward0>)


In [24]:
predictions = logits.argmax(dim=1)

print("Predictions:")
print(predictions[:10])

print("\nActual labels:")
print(labels[:10])

Predictions:
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0')

Actual labels:
tensor([0, 1, 0, 0, 1, 1, 0, 1, 0, 0], device='cuda:0')


In [25]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [26]:
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for input_ids, labels in loader:

        input_ids = input_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        predictions = logits.argmax(dim=1)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples

    accuracy = total_correct / total_samples

    return avg_loss, accuracy

In [27]:
@torch.no_grad()
def evaluate(model, loader, criterion):

    model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for input_ids, labels in loader:

        input_ids = input_ids.to(device)
        labels = labels.to(device)

        logits = model(input_ids)

        loss = criterion(
            logits,
            labels
        )

        total_loss += loss.item() * labels.size(0)

        predictions = logits.argmax(dim=1)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples

    accuracy = total_correct / total_samples

    return avg_loss, accuracy

In [28]:
EPOCHS = 5

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    test_loss, test_acc = evaluate(
        model,
        test_loader,
        criterion
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.4f}"
    )

Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\torch\nn\modules\transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Epoch 1/5 | Train Loss: 0.5851 | Train Acc: 0.6827 | Test Loss: 0.5074 | Test Acc: 0.7469
Epoch 2/5 | Train Loss: 0.4563 | Train Acc: 0.7832 | Test Loss: 0.4719 | Test Acc: 0.7753
Epoch 3/5 | Train Loss: 0.3956 | Train Acc: 0.8196 | Test Loss: 0.4615 | Test Acc: 0.7816
Epoch 4/5 | Train Loss: 0.3460 | Train Acc: 0.8515 | Test Loss: 0.5268 | Test Acc: 0.7710
Epoch 5/5 | Train Loss: 0.3007 | Train Acc: 0.8739 | Test Loss: 0.4942 | Test Acc: 0.7864


In [32]:
def predict_sentiment(text):

    model.eval()

    token_ids = encode_text(text)

    input_ids = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        logits = model(input_ids)

        prediction = logits.argmax(
            dim=1
        ).item()

    if prediction == 1:
        print(logits)
        print('Positive')
    else:
        print(logits)
        print('Negative')

In [34]:
reviews = [
    "This movie was absolutely fantastic and I loved every minute of it.",
    "The movie was boring, poorly written, and a complete waste of time.",
    "Amazing acting and a wonderful story.",
    "I hated this movie. It was terrible."
]

for review in reviews:
    print(review)
    prediction = predict_sentiment(review)
    print()

This movie was absolutely fantastic and I loved every minute of it.
tensor([[ 0.3498, -0.3270]], device='cuda:0')
Negative

The movie was boring, poorly written, and a complete waste of time.
tensor([[ 3.3013, -3.7558]], device='cuda:0')
Negative

Amazing acting and a wonderful story.
tensor([[-2.4580,  3.0120]], device='cuda:0')
Positive

I hated this movie. It was terrible.
tensor([[ 1.5420, -2.0650]], device='cuda:0')
Negative

